In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import harpy as hp

In [ ]:
from spatialdata import read_zarr

sdata=read_zarr( "/Users/arnedf/VIB/DATA/test_data_ilastik/sdata_channels.zarr" )
sdata

see https://github.com/ilastik/ilastik/blob/54e17482cfe8c186a05b367450c4661792f7048c/ilastik/plugins_default/vigra_objfeats.py#L104 for all features ilastik extracts.

we probably want to support:

image + label:
- 'sum' 
- 'mean'
- 'var'
- 'kurtosis'
- 'skew'
- 'min'
- 'max'
- 'quantiles'

label:
- 'area'
- center_of_mass
- radii and axes

These are all implemented in `RasterAggregator`

In [5]:
import dask.array as da

mask=sdata[ "mask_whole_testing" ].data[ None, ... ] # (z,y,x)

image=da.concatenate([ sdata[ _image_name ].data for _image_name in [*sdata.images] ])
image=image[ :, None, ... ] # ( c,z,y,x )

In [6]:
from harpy.utils._aggregate import RasterAggregator

aggregator=RasterAggregator( mask_dask_array=mask, image_dask_array=image)

In [ ]:
aggregator.aggregate_radii_and_axes( depth=100 ).head()

In [ ]:
quantiles=aggregator.aggregate_quantiles(depth=100 ) # gives you a list of dataframes
quantiles[2].head()

In [ ]:
dfs=aggregator.aggregate_stats( stats_funcs=("sum", "mean", "count", "var", "kurtosis") )

In [ ]:
dfs[0] #-> sum, for each object, and each channel in image

In [ ]:
sdata[ "annotation" ].data

In [ ]:
from ilastik.napari.utils import get_annotation

annotated_cells_id, annotation=get_annotation( array_1=sdata[ "annotation" ].data, array_2=sdata[ "mask_whole_testing" ].data )

print( annotated_cells_id )
print(annotation)

In [14]:
# for simplicity first try implementing object classification only using mean intensity
features=aggregator.aggregate_stats( stats_funcs=( "mean" ) ) # retuns a list of dataframes, take the first on (mean intensity)
features[0][ [ 0, 1, "cell_ID" ] ] # only take mean, and only the first two channels
features=features[0][ [ 0, 1, "cell_ID" ] ]
features=features[  features[ "cell_ID" ]!=0 ] # remove features for background

In [ ]:
features  # TODO: write code to create one big dataframe with all extracted features

In [ ]:
X_train=features[ features[ "cell_ID" ].isin( annotated_cells_id )]  # train on these
# drop the cell_ID column
X_train=X_train.drop( "cell_ID", axis=1 )
X_train.head()

In [ ]:
annotation

In [ ]:
annotated_cells_id

In [ ]:
X_train

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, annotation)
y_pred = clf.predict(X_train)
y_pred

In [ ]:
# run on all data
y_pred_all=clf.predict( features.drop( [ "cell_ID" ], axis=1 ) )  # gives us a prediction for every cell
y_pred_all.shape

In [ ]:
y_pred_all[:10]  # this is a label for every mask

In [ ]:
cell_ids=features[ "cell_ID" ].values # cell_IDs
cell_ids
cell_ids[ :10 ]

In [24]:
# now generate the relabeld mask efficiently
mask=sdata[ "mask_whole_testing" ].data # relabel this

In [ ]:
import numpy as np
import dask.array as da

# create the relabeld mask as a dask array

assert cell_ids.shape == y_pred_all.shape

max_id = cell_ids.max()
lookup = np.zeros(max_id + 1, dtype=y_pred_all.dtype)
lookup[cell_ids] = y_pred_all 
relabelled_masks = da.take(lookup, mask) # maps each cell_id to its new label

In [ ]:
from spatialdata.models import Labels2DModel

se= Labels2DModel.parse( relabelled_masks )

sdata[  "predicted_labels" ] = se

In [ ]:
from napari_spatialdata import Interactive

Interactive( sdata )

In [ ]:
# dummy code to explain the working of np.take
import numpy as np

lookup = np.array([0, 10, 20, 30, 40])

mask = np.array([
    [3, 1, 2],
    [3, 4, 0]
])

result = np.take(lookup, mask)
result